# ReAct Pattern - Thought, Action, Observation


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2fd02b2eaa-16c3-4f92-8f97-06329fbcccd4_716x550-7.gif" alt="ReAct pattern" width="500"/>

This notebook uses the ReAct Pattern to analyze a staged mobile-forensics case about a Signal attachment attempt during a reported unattended interval. Instead of generic demo tools, you will work with the same local evidence package described in `02_case_overview.md`.

ReAct is a bounded workflow for answering a question through repeated **thought -> action -> observation -> response** cycles. In Lab 3, the `Environment` is the staged forensic evidence package, which the agent accesses indirectly through tools that read artifact files such as `incident_window.csv`, `messaging_events.csv`, and `network_events.csv`.

In this lab, the goal is not to guess whether a message was delivered. The goal is to reason one step at a time, inspect each observation, and produce a conclusion that stays inside the evidence.

This is the **third pattern lab** in the course. If you want to revisit the earlier lessons first:

* [Reflection Pattern](../lab1_reflection_pattern/03_lab_notebook.ipynb)
* [Tool Use Pattern](../lab2_tool_use_pattern/03b_lab_notebook.ipynb)


## Lab Question and Response Format

Use the staged artifacts to answer this case question:

**Did the Signal attachment attempt happen during the reported unattended interval, and what does the network evidence allow us to say about delivery?**

Return the same four-part response format in both parts of the notebook:

1. `ReAct step log`
2. `unattended-interval evidence`
3. `Signal attachment timing evidence`
4. `connectivity context and evidence-bounded conclusion`

Suggested layout for the final answer:
- Under `ReAct step log`, use a numbered list with one short line per tool step.
- Under `unattended-interval evidence`, list the start and end timestamps.
- Under `Signal attachment timing evidence`, list the attempt timestamp and whether it falls inside the interval.
- Under `connectivity context and evidence-bounded conclusion`, list the network restore time and then write one careful conclusion sentence.

Required observation sequence for the guided walkthrough:
`get_incident_window -> get_message_attempt(app="Signal") -> get_network_restore_time`

Important vocabulary for this lab:
- `tool`: a small helper function that reads one evidence file and returns a result.
- `tool call`: the model's request to run one tool.
- `JSON`: a structured text format using names and values, such as `"start": "2026-02-20T14:10:00Z"`.
- `observation`: the result returned by a tool call.
- `evidence-bounded answer`: an answer that says only what the observed records support.

Run the notebook from top to bottom. You are not expected to write new Python for the guided walkthrough; your main task is to explain how each observation supports the next step in the ReAct loop.


### Step 1: Set Up the Notebook


Run this setup cell first. It loads the lab configuration from `.env`, checks that you opened the notebook from the correct folder, connects to the model client, and points the notebook to the Lab 3 data files.

To run a code cell, click inside the cell and press `Shift+Enter`, or use the notebook's Run button. Run cells in order so each later cell has the variables and tools created by earlier cells.

You do not need to understand every Python line here. If this cell fails, fix the setup issue before moving on.


In [1]:
import csv
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab3_react_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 3 data folder')

from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.utils.extraction import extract_tag_content

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)


Repo root: /home/frank/projects/agentic-AI4-forensics
Lab folder: /home/frank/projects/agentic-AI4-forensics/lab3_react_pattern


### Step 2: Review the Staged Artifact Package

The lab works with local evidence only. Before you call any tools, inspect the case manifest so you know the incident window, acquisition details, and artifact files you are about to query.


In [2]:
# Load the manifest, which is the summary file for this staged evidence package.
case_manifest = json.loads((data_dir / 'artifact_manifest.json').read_text(encoding='utf-8'))

# Show the files in the data folder in a stable order so every student sees the same list.
artifact_files = sorted(path.name for path in data_dir.iterdir() if path.is_file())

# Display only the case details students need before using the tools.
{
    "case_id": case_manifest["case_id"],
    "device": case_manifest["device"],
    "incident_window_utc": case_manifest["incident_window_utc"],
    "interval_source": case_manifest["interval_source"],
    "artifact_files": artifact_files,
}


{'case_id': 'RA-2026-009',
 'device': {'platform': 'Android',
  'model': 'Samsung Galaxy A54',
  'os_version': 'Android 14'},
 'incident_window_utc': {'start': '2026-02-20T14:10:00Z',
  'end': '2026-02-20T14:25:00Z'},
 'interval_source': 'staff observation log',
 'artifact_files': ['artifact_manifest.json',
  'chain_of_custody.csv',
  'incident_window.csv',
  'messaging_events.csv',
  'network_events.csv']}

## Part 1: Walk the ReAct Workflow Manually

In Part 1, you define the forensic tools, inspect the ReAct prompt, and run the loop one turn at a time. This keeps the model's thought, tool call, observation, and final response visible before you hand the same workflow to `ReactAgent`.


### Step 3: Define the Forensic Tools

These tools expose one narrow part of the mini-case at a time so the ReAct loop has to inspect the evidence step by step.


This cell defines the three forensic tools used in the lab. Each tool reveals one small part of the case so the model has to gather evidence step by step instead of guessing.

In the examples below, the text inside parentheses is the input. Empty parentheses mean the tool does not need any extra input. The output is the evidence returned by the tool.

Tools introduced here, with simple examples:

- `get_incident_window()`

  Example output:

  ```json
  {
    "start": "2026-02-20T14:10:00Z",
    "end": "2026-02-20T14:25:00Z",
    "start_source": "staff_observation",
    "end_source": "staff_observation",
    "start_note": "phone left on supply table before coordinator stepped away",
    "end_note": "phone recovered from supply table when coordinator returned"
  }
  ```

- `get_message_attempt(app="Signal")`

  Example output:

  ```json
  {
    "timestamp_utc": "2026-02-20T14:16:11Z",
    "app": "Signal",
    "event": "attachment_attempt",
    "contact": "+1-555-0179",
    "details": "outgoing image attachment attempt recorded while mobile data unavailable"
  }
  ```

- `get_network_restore_time()`
  Example output: `"2026-02-20T14:28:02Z"`


In [3]:
# Helper function: read one CSV evidence file and return its rows.
# Each row becomes a small dictionary, such as {"timestamp_utc": "..."}.
def read_csv(filename: str) -> list[dict]:
    with (data_dir / filename).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


# Tool 1: return the reported start and end of the unattended interval.
# The ReAct loop needs this first so later timestamps have a time window to compare against.
@tool
def get_incident_window() -> str:
    """Return the start and end timestamps for the reported unattended interval."""
    rows = read_csv('incident_window.csv')
    # Pick the row that marks when the phone became unattended.
    start = next(row for row in rows if row['event'] == 'unattended_interval_start')
    # Pick the row that marks when the phone was recovered.
    end = next(row for row in rows if row['event'] == 'unattended_interval_end')
    return json.dumps(
        {
            'start': start['timestamp_utc'],
            'end': end['timestamp_utc'],
            'start_source': start['source'],
            'end_source': end['source'],
            'start_note': start['notes'],
            'end_note': end['notes'],
        }
    )


# Tool 2: return the Signal attachment-attempt record.
# This is the event we compare against the unattended interval.
@tool
def get_message_attempt(app: str) -> str:
    """Return the first recorded messaging attachment attempt for the requested app."""
    rows = read_csv('messaging_events.csv')
    # Find the first messaging record whose app name matches the requested app.
    match = next((row for row in rows if row['app'].lower() == app.lower()), None)
    if match is None:
        return json.dumps({'error': f'No messaging event found for {app}.'})
    return json.dumps(match)


# Tool 3: return when mobile data came back online.
# This does not prove delivery; it helps keep the final claim evidence-bounded.
@tool
def get_network_restore_time() -> str:
    """Return the timestamp when mobile data connectivity was restored."""
    rows = read_csv('network_events.csv')
    # Find the network row where the status changed back to restored.
    restored = next(row for row in rows if row['status'] == 'restored')
    return restored['timestamp_utc']


# Store the tools by name so the ReAct loop can run the tool the model asks for.
available_tools = {
    'get_incident_window': get_incident_window,
    'get_message_attempt': get_message_attempt,
    'get_network_restore_time': get_network_restore_time,
}


### Step 4: Inspect the Tool Schemas

Before you run the ReAct loop, inspect the generated tool signatures. These schemas are the tool descriptions the model will use when deciding which action to take next.


In [4]:
tool_schema_preview = {
    get_incident_window.name: json.loads(get_incident_window.fn_signature),
    get_message_attempt.name: json.loads(get_message_attempt.fn_signature),
    get_network_restore_time.name: json.loads(get_network_restore_time.fn_signature),
}

tool_schema_preview


{'get_incident_window': {'name': 'get_incident_window',
  'description': 'Return the start and end timestamps for the reported unattended interval.',
  'parameters': {'properties': {}}},
 'get_message_attempt': {'name': 'get_message_attempt',
  'description': 'Return the first recorded messaging attachment attempt for the requested app.',
  'parameters': {'properties': {'app': {'type': 'str'}}}},
 'get_network_restore_time': {'name': 'get_network_restore_time',
  'description': 'Return the timestamp when mobile data connectivity was restored.',
  'parameters': {'properties': {}}}}

### Step 5: Define the ReAct System Prompt


This cell defines the system prompt that teaches the model how to behave in a ReAct loop.

Pay attention to the four required stages: `thought`, `action`, `observation`, and `response`. The prompt also restricts the model to one tool call at a time.

The angle-bracket labels, such as `<thought>` and `<tool_call>`, are just markers that make the model's output easier to separate into parts. Students do not need to memorize the syntax; focus on the pattern: reason, act, observe, then decide whether to continue.


In [5]:
REACT_SYSTEM_PROMPT = """
## Role
You are a function-calling AI model that reasons and acts in a loop until you can deliver a final response to the user.

## Loop Structure
Each iteration of your loop consists of four stages:

1. Thought — reason about what you need next. Wrap in <thought></thought> tags.
2. Action — call one tool in <tool_call></tool_call> tags.
3. Observation — the tool result will be returned in <observation></observation> tags.
4. Response — when you have enough evidence, answer in <response></response> tags.

## Tool Call Format
<tool_call>
{"name": <function-name>, "arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

## Available Tools
<tools>
%s
</tools>

## Additional Constraints
- Use exactly one tool call at a time.
- Do not skip directly to a final answer when evidence is still missing.
- Keep the final answer evidence-bounded.
"""


### Step 6: Prepare the Manual ReAct Walkthrough

For the manual walkthrough, we will use a prompt that forces a predictable tool order so you can inspect each turn clearly.
As you run each step, pay attention to four things: the question being answered, the tool call chosen, the observation returned, and why that observation changes the next step.
Do not jump to the final answer until the needed observations have been collected.

Use this simple note-taking template while you run the cells:

| Step | Tool Called | Observation Returned | What This Observation Lets Us Decide |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |


This cell prepares the manual walkthrough. It builds the tool list, states the forensic question, starts the conversation history, and defines a helper function that runs one ReAct turn.

This is the same short-term memory idea from `03a_memory_demo.ipynb`: `chat_history` stores the question, model messages, and tool observations so the next turn can build on the previous turn.

This is the core mechanism of the lab: ask a question, make one tool call, inspect the observation, then decide what should happen next.


In [6]:
# Combine the three tool descriptions into one block the model can read.
# This is the model's "menu" of available actions.
tools_signature = ',\n'.join(
    [
        get_incident_window.fn_signature,
        get_message_attempt.fn_signature,
        get_network_restore_time.fn_signature,
    ]
)
# Insert that tool menu into the ReAct system prompt from Step 5.
formatted_system_prompt = REACT_SYSTEM_PROMPT % tools_signature

# This is the case question the model will answer by using one tool at a time.
USER_QUESTION = (
    'Determine whether the Signal attachment attempt happened during the reported unattended interval ' 
    'and use network restoration evidence to avoid overstating confirmed delivery. ' 
    'Use exactly one tool call per turn in this order: get_incident_window, ' 
    'then get_message_attempt for Signal, then get_network_restore_time. ' 
    'After the third observation, provide a final answer using this exact structure: ' 
    'ReAct step log: a numbered list with one short line for each tool step; ' 
    'unattended-interval evidence: list the start and end timestamps; ' 
    'Signal attachment timing evidence: list the attempt timestamp and say whether it is inside the interval; ' 
    'connectivity context and evidence-bounded conclusion: list the network restore time and then write one careful conclusion sentence that does not claim confirmed delivery.'
)

# Start the conversation history.
# The system message gives the rules; the user message gives the case question.
chat_history = [
    {'role': 'system', 'content': formatted_system_prompt},
    {'role': 'user', 'content': f'<question>{USER_QUESTION}</question>'},
]


# Run one ReAct turn: model thinks, chooses a tool, receives an observation.
def run_single_react_turn(history: list[dict]):
    # Ask the model what to do next using the conversation so far.
    output = client.chat.completions.create(messages=history, model=MODEL).choices[0].message.content
    print(output)
    # Save the model's thought/action message so the next turn has context.
    history.append({'role': 'assistant', 'content': output})

    # Look inside the model output for a <tool_call>...</tool_call> block.
    tool_call = extract_tag_content(output, tag='tool_call')
    if not tool_call.found:
        # If there is no tool call, the model may already be trying to answer.
        return output, None, None

    # Convert the tool-call JSON text into a Python dictionary.
    tool_call_dict = json.loads(tool_call.content[0])
    # Use the tool name from the model to pick and run the matching local tool.
    tool_result = available_tools[tool_call_dict['name']].run(**tool_call_dict['arguments'])
    print('\nTool call:', tool_call_dict)
    print('Tool result:', tool_result)
    # Feed the tool result back as an observation for the next ReAct turn.
    history.append({'role': 'user', 'content': f'<observation>{tool_result}</observation>'})
    return output, tool_call_dict, tool_result


### Step 7: Get the Incident Window


This is the first turn of the manual ReAct loop. Run it and inspect what the model chooses to do first.

What to look for: the model should identify that it first needs the unattended interval before it can judge whether the Signal event happened inside or outside that window.

After running this cell, write down the interval start time and end time. You should see an observation showing `14:10:00 UTC` as the start and `14:25:00 UTC` as the end.


In [7]:
output_1, tool_call_1, tool_result_1 = run_single_react_turn(chat_history)


<tool_call>
{"name": "get_incident_window", "arguments": {}, "id": 1}
</tool_call>

Tool call: {'name': 'get_incident_window', 'arguments': {}, 'id': 1}
Tool result: {"start": "2026-02-20T14:10:00Z", "end": "2026-02-20T14:25:00Z", "start_source": "staff_observation", "end_source": "staff_observation", "start_note": "phone left on supply table before coordinator stepped away", "end_note": "phone recovered from supply table when coordinator returned"}


### Step 8: Get the Signal Attachment Attempt


This is the second turn of the manual ReAct loop. Run it after reviewing the first observation.

What to look for: the next tool call should focus on the Signal attachment attempt, because the model now knows the time window and can compare the message timestamp against it.

After running this cell, write down the Signal timestamp and decide whether it falls inside the interval from Step 1. You should see an attachment attempt at `14:16:11 UTC`, which is between `14:10:00 UTC` and `14:25:00 UTC`.


In [8]:
output_2, tool_call_2, tool_result_2 = run_single_react_turn(chat_history)


<tool_call>
{"name": "get_message_attempt", "arguments": {"app": "Signal"}, "id": 2}
</tool_call>

Tool call: {'name': 'get_message_attempt', 'arguments': {'app': 'Signal'}, 'id': 2}
Tool result: {"timestamp_utc": "2026-02-20T14:16:11Z", "app": "Signal", "event": "attachment_attempt", "contact": "+1-555-0179", "details": "outgoing image attachment attempt recorded while mobile data unavailable"}


### Step 9: Check Network Restoration Context


This is the third turn of the manual ReAct loop. Run it after reviewing the message observation.

What to look for: the model should now check when connectivity returned so it can keep the conclusion evidence-bounded and avoid treating an attachment attempt as confirmed delivery.

After running this cell, compare the reconnect time to the interval end time. You should see that mobile data returned at `14:28:02 UTC`, which is after the unattended interval ended at `14:25:00 UTC`.


In [9]:
output_3, tool_call_3, tool_result_3 = run_single_react_turn(chat_history)


<tool_call>
{"name": "get_network_restore_time", "arguments": {}, "id": 3}
</tool_call>

Tool call: {'name': 'get_network_restore_time', 'arguments': {}, 'id': 3}
Tool result: 2026-02-20T14:28:02Z


### Step 10: Ask the Model for the Evidence-Bounded Response


This cell asks the model for a final response after the needed observations have been collected.

A good final answer should cite the observed timestamps, answer the main timing question, and use the connectivity evidence to avoid claiming more than the records support.

A useful mini-template is:
`ReAct step log:`
`1. Checked ...`
`2. Checked ...`
`3. Checked ...`
`unattended-interval evidence:` start ..., end ...
`Signal attachment timing evidence:` attempt ..., inside/outside interval ...
`connectivity context and evidence-bounded conclusion:` network restored ..., therefore the evidence supports ... but does not confirm ...

For this lab, be careful not to say the attachment was definitely delivered. The evidence shows an attempt and a later reconnect time, not confirmed delivery.


In [10]:
final_output = client.chat.completions.create(messages=chat_history, model=MODEL).choices[0].message.content
print(final_output)


<response>
ReAct step log:
1. Retrieved unattended-interval evidence: 2026-02-20T14:10:00Z to 2026-02-20T14:25:00Z
2. Found Signal attachment timing evidence: 2026-02-20T14:16:11Z (within unattended interval)
3. Identified connectivity context: Network restored at 2026-02-20T14:28:02Z (after attachment attempt)

Conclusion: The Signal attachment attempt occurred during the unattended interval, but mobile data was unavailable at the time. Network restoration occurred 2 minutes and 51 seconds after the attempt, indicating delivery confirmation cannot be assured without further evidence.
</response>


## Part 2: Run the Same Workflow with `ReactAgent`

In Part 2, you hand the same three forensic tools to `ReactAgent`. The case question stays the same, but the agent now manages the thought-action-observation loop for you.


### Step 11: Hand the Tools to `ReactAgent`

This cell creates the packaged `ReactAgent` using the same three tools from the manual walkthrough.

Why this matters: students can compare the explicit step-by-step process they just inspected with a wrapped version that automates the same pattern.


In [11]:
from agentic_patterns.react_pattern.react_agent import ReactAgent

react_agent = ReactAgent(
    tools=[get_incident_window, get_message_attempt, get_network_restore_time],
    client=client,
    model=MODEL,
)


### Step 12: Run `ReactAgent` on the Same Case

This cell runs the packaged `ReactAgent` on the same forensic question.

Compare its behavior to your manual walkthrough: what tool did it choose first, what observations did it rely on, and did its final answer stay evidence-bounded?

Reflection questions to answer after the run:
- Did the agent gather the same three observations you gathered manually?
- Did the agent use the observations in a logical order?
- Did the agent avoid saying the attachment was delivered?
- If the agent made a claim that was too strong, what would you revise?


In [12]:
print(react_agent.run(user_msg=USER_QUESTION))


## Manual vs. Packaged ReAct

`Part 1` made each ReAct turn explicit: the model thought about the next need, called one tool, received one observation, and used that observation to decide the next step. `Part 2` handed the same tools and question to `ReactAgent`, which managed the loop automatically. In both versions, the final answer should stay bounded: the records support an in-window Signal attachment attempt, but they do not prove confirmed delivery.
